## Notebook for using the CDS API to get ERA5 data

#### Run the pip install to get the latest cdsapi

In [1]:
#!pip install cdsapi
!pip install --upgrade cdsapi
# created .cdsapirc file in ~/  per https://ads-beta.atmosphere.copernicus.eu/how-to-api

  Attempting uninstall: cads-api-client
    Found existing installation: cads-api-client 1.0.3
    Uninstalling cads-api-client-1.0.3:
      Successfully uninstalled cads-api-client-1.0.3
  Attempting uninstall: cdsapi
    Found existing installation: cdsapi 0.7.0
    Uninstalling cdsapi-0.7.0:
      Successfully uninstalled cdsapi-0.7.0


In [2]:
import cdsapi
import requests

### This block is for grabbing a single year, entire globe <br><br> This actually errors out and says the request is too large for netcdf, apparently

In [2]:
c = cdsapi.Client()

c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': [
            #'2m_temperature', 
            'land_sea_mask',
        ],
        'year': '2023',
        'month': [
            #'01', '02', '03',
            #'04', '05', '06',
            #'07', '08', 
            '09',
            #'10', '11', '12',
        ],
        'day': [
            '01', #'02', '03',
            # '04', '05', '06',
            # '07', '08', '09',
            # '10', '11', '12',
            # '13', '14', '15',
            # '16', '17', '18',
            # '19', '20', '21',
            # '22', '23', '24',
            # '25', '26', '27',
            # '28', '29', '30',
            # '31',
        ],
        'time': [
            # '00:00', '01:00', '02:00',
            # '03:00', '04:00', '05:00',
            # '06:00', '07:00', '08:00',
            # '09:00', '10:00', '11:00',
             '12:00', #'13:00', '14:00',
            # '15:00', '16:00', '17:00',
            # '18:00', '19:00', '20:00',
            # '21:00', '22:00', '23:00',
        ],
        'format': 'netcdf',
    },
    'ERA5-2023-09-01-Full-LSM-download.nc')

2024-07-04 15:47:39,562 INFO Welcome to the CDS
2024-07-04 15:47:39,562 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/resources/reanalysis-era5-single-levels
2024-07-04 15:47:39,841 INFO Request is queued
2024-07-04 15:47:42,704 INFO Request is completed
2024-07-04 15:47:42,705 INFO Downloading https://download-0018.copernicus-climate.eu/cache-compute-0018/cache/data5/adaptor.mars.internal-1720133261.933593-1638-18-449eb970-0a03-4b82-b836-fe806558723e.nc to ERA5-2023-09-01-Full-LSM-download.nc (2M)
2024-07-04 15:47:44,709 INFO Download rate 1016.7K/s                            


Result(content_length=2086240,content_type=application/x-netcdf,location=https://download-0018.copernicus-climate.eu/cache-compute-0018/cache/data5/adaptor.mars.internal-1720133261.933593-1638-18-449eb970-0a03-4b82-b836-fe806558723e.nc)

### This block is for specific years and months, specific area lat/lon

In [ ]:
# try multiple years, same region, JJA
c = cdsapi.Client()

c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': [
            '2m_temperature', 'land_sea_mask',
        ],
        'year': ['2020','2021','2022','2023'],
        'month': [
            '06', '07', '08',
        ],
        'day': [
            '01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12',
            '13', '14', '15',
            '16', '17', '18',
            '19', '20', '21',
            '22', '23', '24',
            '25', '26', '27',
            '28', '29', '30',
            '31',
        ],
        'time': [
            '00:00', '01:00', '02:00',
            '03:00', '04:00', '05:00',
            '06:00', '07:00', '08:00',
            '09:00', '10:00', '11:00',
            '12:00', '13:00', '14:00',
            '15:00', '16:00', '17:00',
            '18:00', '19:00', '20:00',
            '21:00', '22:00', '23:00',
        ],
        'area': [
            50, -124, 47,
            -122,
        ],
        'format': 'netcdf',
    },
    '2020-2023_JJA_PugetSound_T2m-LSM.nc')

### <br> Using the Daily Statistics Application to get daily means instead of pulling hourly data and calculating it <br> https://cds.climate.copernicus.eu/cdsapp#!/software/app-c3s-daily-era5-statistics?tab=overview <br><br> From this forum post: https://forum.ecmwf.int/t/retrieve-daily-era5-era5-land-data-using-the-cds-api/1150/1 <br><br> After running - note each month is roughly 128MB or about 1.5GB per year! <br> And, it took about 10 min total to retrieve 12 months (12 files)

In [9]:
import cdsapi
import requests
 
# CDS API script to use CDS service to retrieve daily ERA5* variables and iterate over
# all months in the specified years.
 
# Requires:
# 1) the CDS API to be installed and working on your system
# 2) You have agreed to the ERA5 Licence (via the CDS web page)
# 3) Selection of required variable, daily statistic, etc
 
# Output:
# 1) separate netCDF file for chosen daily statistic/variable for each month
 
c = cdsapi.Client()#timeout=300)
 
# Uncomment years as required
 
years =  [
#    '1960',
#    '1961','1962','1963','1964'
#   '1965','1966',
#    '1967',
#    '1968',
    '1969',
           '1970',
   '1971','1972','1973','1974','1975',
   '1976','1977','1978','1979',
           '1980', 
   '1981','1982', '1983', '1984',
           '1985',
'1986', '1987','1988', '1989', 
   '1990'#,
#            '1991', '1992', '1993',
#            '1994', '1995', '1996',
#            '1997', '1998', '1999'#,
#           '2000',
  #  '2001', '2002',
  #          '2003', '2004', '2005',
  #          '2006', '2007', '2008',
  #          '2009', 
  # '2010', 
  #  '2011',
  #           '2012', '2013', '2014',
  #           '2015', '2016', '2017',
  #           '2018', '2019'#, '2020'
  #          '2021',
  #           '2022'
  #  '2023',
  #   '2024'
]
 
 
# Retrieve all months for a given year.
 
months = ['01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12']
 
# For valid keywords, see Table 2 of:
# https://datastore.copernicus-climate.eu/documents/app-c3s-daily-era5-statistics/C3S_Application-Documentation_ERA5-daily-statistics-v2.pdf
 
# select your variable; name must be a valid ERA5 CDS API name.
var = "sea_surface_temperature" #"2m_temperature"
 
# Select the required statistic, valid names given in link above
stat = "daily_mean"
 
# Loop over years and months
 
for yr in years:
    for mn in months:
        result = c.service(
        "tool.toolbox.orchestrator.workflow",
        params={
             "realm": "user-apps",
             "project": "app-c3s-daily-era5-statistics",
             "version": "master",
             "kwargs": {
                 "dataset": "reanalysis-era5-single-levels",
                 "product_type": "reanalysis",
                 "variable": var,
                 "statistic": stat,
                 "year": yr,
                 "month": mn,
                 "time_zone": "UTC+00:0",
                 "frequency": "1-hourly",
#
# Users can change the output grid resolution and selected area
#
#                "grid": "1.0/1.0",
#                "area":{"lat": [10, 60], "lon": [65, 140]}
 
                 },
        "workflow_name": "application"
        })
         
# set name of output file for each month (statistic, variable, year, month     
 
        file_name = "../../../Data/ERA5-global/Baseline/" + yr + "/download_" + stat + "_" + var + "_" + yr + "_" + mn + ".nc"
         
        location=result[0]['location']
        res = requests.get(location, stream = True)
        print("Writing data to " + file_name)
        with open(file_name,'wb') as fh:
            for r in res.iter_content(chunk_size = 1024):
                fh.write(r)
        fh.close()

2024-08-28 09:21:24,065 INFO Welcome to the CDS.
 As per our announcements on the Forum, this instance of CDS will soon be decommissioned.
 Please update your cdsapi package to a version >=0.7.0, create an account on CDS-Beta and update your .cdsapirc file. We strongly recommend users to check our Guidelines at https://confluence.ecmwf.int/x/uINmFw
 The current legacy system will be kept for a while, but we will reduce resources gradually until full decommissioning in September 2024.
2024-08-28 09:21:24,066 WARNING MOVE TO CDS-Beta
2024-08-28 09:21:24,067 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-48a0cd94c0ba463bb1c9c22f40e69278
2024-08-28 09:21:24,286 INFO Request is queued
2024-08-28 09:21:25,571 INFO Request is running
2024-08-28 09:25:45,325 INFO Request is completed


Writing data to ../../../Data/ERA5-global/Baseline/1969/download_daily_mean_sea_surface_temperature_1969_01.nc


2024-08-28 09:29:30,100 INFO Welcome to the CDS.
 As per our announcements on the Forum, this instance of CDS will soon be decommissioned.
 Please update your cdsapi package to a version >=0.7.0, create an account on CDS-Beta and update your .cdsapirc file. We strongly recommend users to check our Guidelines at https://confluence.ecmwf.int/x/uINmFw
 The current legacy system will be kept for a while, but we will reduce resources gradually until full decommissioning in September 2024.
2024-08-28 09:29:30,101 WARNING MOVE TO CDS-Beta
2024-08-28 09:29:30,102 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-f566a081f61e470abca960b7a629a3b7
2024-08-28 09:29:30,324 INFO Request is queued
2024-08-28 09:29:31,529 INFO Request is running
2024-08-28 09:32:23,699 INFO Request is completed


Writing data to ../../../Data/ERA5-global/Baseline/1969/download_daily_mean_sea_surface_temperature_1969_02.nc


2024-08-28 09:32:40,724 INFO Welcome to the CDS.
 As per our announcements on the Forum, this instance of CDS will soon be decommissioned.
 Please update your cdsapi package to a version >=0.7.0, create an account on CDS-Beta and update your .cdsapirc file. We strongly recommend users to check our Guidelines at https://confluence.ecmwf.int/x/uINmFw
 The current legacy system will be kept for a while, but we will reduce resources gradually until full decommissioning in September 2024.
2024-08-28 09:32:40,725 WARNING MOVE TO CDS-Beta
2024-08-28 09:32:40,726 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-62736e7a5cf24bad92b7b6e0816ad1a0
2024-08-28 09:32:41,005 INFO Request is queued
2024-08-28 09:32:42,209 INFO Request is running
2024-08-28 09:35:34,824 INFO Request is completed


...

Writing data to ../../../Data/ERA5-global/Baseline/1990/download_daily_mean_sea_surface_temperature_1990_10.nc


2024-08-29 03:09:13,425 INFO Welcome to the CDS.
 As per our announcements on the Forum, this instance of CDS will soon be decommissioned.
 Please update your cdsapi package to a version >=0.7.0, create an account on CDS-Beta and update your .cdsapirc file. We strongly recommend users to check our Guidelines at https://confluence.ecmwf.int/x/uINmFw
 The current legacy system will be kept for a while, but we will reduce resources gradually until full decommissioning in September 2024.
2024-08-29 03:09:13,426 WARNING MOVE TO CDS-Beta
2024-08-29 03:09:13,427 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-317c5e1fad6f4d259a8edfc4f8e647c9
2024-08-29 03:09:13,662 INFO Request is queued
2024-08-29 03:09:14,873 INFO Request is running
2024-08-29 03:12:07,230 INFO Request is completed


Writing data to ../../../Data/ERA5-global/Baseline/1990/download_daily_mean_sea_surface_temperature_1990_11.nc


2024-08-29 03:12:51,527 INFO Welcome to the CDS.
 As per our announcements on the Forum, this instance of CDS will soon be decommissioned.
 Please update your cdsapi package to a version >=0.7.0, create an account on CDS-Beta and update your .cdsapirc file. We strongly recommend users to check our Guidelines at https://confluence.ecmwf.int/x/uINmFw
 The current legacy system will be kept for a while, but we will reduce resources gradually until full decommissioning in September 2024.
2024-08-29 03:12:51,528 WARNING MOVE TO CDS-Beta
2024-08-29 03:12:51,529 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-d85839a69b7149018beae28c809de515
2024-08-29 03:12:51,753 INFO Request is queued
2024-08-29 03:12:52,943 INFO Request is running
2024-08-29 03:12:57,088 INFO Request is completed


Writing data to ../../../Data/ERA5-global/Baseline/1990/download_daily_mean_sea_surface_temperature_1990_12.nc


## 

In [2]:
!ls -al "../../../Data/ERA5-global/Analysis/2024/"

total 1622800
drwxr-xr-x@  9 tedscott  staff        288 Jul 23 15:24 .
drwxr-xr-x@ 40 tedscott  staff       1280 Jul  4 19:09 ..
-rw-r--r--@  1 tedscott  staff  128778531 Jul 23 15:15 download_daily_mean_2m_temperature_2024_01.nc
-rw-r--r--@  1 tedscott  staff  120472595 Jul 23 15:18 download_daily_mean_2m_temperature_2024_02.nc
-rw-r--r--@  1 tedscott  staff  128778532 Jul 23 15:22 download_daily_mean_2m_temperature_2024_03.nc
-rw-r--r--@  1 tedscott  staff  124625564 Jul 23 15:23 download_daily_mean_2m_temperature_2024_04.nc
-rw-r--r--@  1 tedscott  staff  128778534 Jul 23 15:23 download_daily_mean_2m_temperature_2024_05.nc
-rw-r--r--@  1 tedscott  staff  124625566 Jul 23 15:23 download_daily_mean_2m_temperature_2024_06.nc
-rw-r--r--@  1 tedscott  staff   74789948 Jul 23 15:23 download_daily_mean_2m_temperature_2024_07.nc


## Get Satellite Ocean Color Data for Salome and Risako

In [2]:
import cdsapi
import requests

# try multiple years, specific region including Galapagos
c = cdsapi.Client()

c.retrieve(
    'satellite-ocean-colour',
    {
        'version': '6_0',
        'format': 'zip',
        'variable': [
            'mass_concentration_of_chlorophyll_a',
        ],
        'projection': 'regular_latitude_longitude_grid',
        'year': ['2019',
                 #'2020','2021','2022','2023',
                ],
        'month': [
            '01', '02', '03','04','05','06','07','08','09','10','11','12',
        ],
        'day': [
            '01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12',
            '13', '14', '15',
            '16', '17', '18',
            '19', '20', '21',
            '22', '23', '24',
            '25', '26', '27',
            '28', '29', '30',
            '31',
        ],
        'area': [
            2, -92, -2,
            -88,
        ],
        'format': 'netcdf',
    },
    '2019-2023_OceanColour.nc')

2024-07-09 15:25:15,108 INFO Welcome to the CDS
2024-07-09 15:25:15,108 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/resources/satellite-ocean-colour
2024-07-09 15:25:15,301 INFO Request is queued
2024-07-09 15:25:16,480 INFO Request is failed
2024-07-09 15:25:16,481 ERROR Message: the request you have submitted is not valid
2024-07-09 15:25:16,481 ERROR Reason:  Request too large. Requesting 90 items, limit is 31
2024-07-09 15:25:16,482 ERROR   Traceback (most recent call last):
2024-07-09 15:25:16,483 ERROR     File "/opt/cds/cdsinf/python/lib/cdsinf/runner/dispatcher.py", line 163, in _consume
2024-07-09 15:25:16,483 ERROR       result = handle_locally()
2024-07-09 15:25:16,483 ERROR     File "/opt/cds/cdsinf/python/lib/cdsinf/runner/dispatcher.py", line 252, in <lambda>
2024-07-09 15:25:16,484 ERROR       lambda: self.handle_exception(context, e),
2024-07-09 15:25:16,484 ERROR     File "/opt/cds/cdsinf/python/lib/cdsinf/runner/dispatcher.py", line 383, in handle

Exception: the request you have submitted is not valid. Request too large. Requesting 90 items, limit is 31.